In [ ]:
import pandas as pd

def load_and_display_heads():
    # File paths (update with your actual file paths)
    game_data_file = '../data/games.csv'
    play_data_file = '../data/plays.csv'
    player_play_data_file = '../data/player_play.csv'
    player_data_file = '../data/players.csv'
    tracking_data_file = '../data/tracking_week_1.csv'  # Example for week 1
    
    # Load datasets
    try:
        game_data = pd.read_csv(game_data_file)
        play_data = pd.read_csv(play_data_file)
        player_play_data = pd.read_csv(player_play_data_file)
        player_data = pd.read_csv(player_data_file)
        tracking_data = pd.read_csv(tracking_data_file)
        
        # Display heads of each dataset
        print("Game Data Head:")
        print(game_data.head())
        print("\nPlay Data Head:")
        print(play_data.head())
        print("\nPlayer Play Data Head:")
        print(player_play_data.head())
        print("\nPlayer Data Head:")
        print(player_data.head())
        print("\nTracking Data Head:")
        print(tracking_data.head())
    
    except FileNotFoundError as e:
        print(f"Error: {e}")
    except pd.errors.EmptyDataError:
        print("Error: One of the files is empty or corrupted.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

    return game_data, play_data, player_play_data, player_data, tracking_data

game_data, play_data, player_play_data, player_data, tracking_data = load_and_display_heads()


In [ ]:
def reduce_tracking_data_direct(tracking_data, delta):
    """
    Reduces tracking data by keeping only rows for frameId = snap frame + 30 for each gameId and playId.
    
    Args:
        tracking_data (pd.DataFrame): Player tracking data with frame-by-frame information.
    
    Returns:
        pd.DataFrame: A reduced DataFrame containing only the relevant rows.
    """
    # Create a multi-index dictionary to map (gameId, playId) to snapFrameId
    snap_frames = tracking_data[tracking_data["frameType"] == "SNAP"]
    snap_frame_dict = snap_frames.set_index(["gameId", "playId"])["frameId"].to_dict()

    # Calculate the target frame for each row in tracking data
    tracking_data["targetFrameId"] = tracking_data.set_index(["gameId", "playId"]).index.map(snap_frame_dict) + delta

    # Filter the rows where frameId matches targetFrameId
    reduced_tracking = tracking_data[tracking_data["frameId"] == tracking_data["targetFrameId"]]

    # Drop the targetFrameId column as it's no longer needed
    reduced_tracking = reduced_tracking.drop(columns=["targetFrameId"])

In [ ]:
def generate_y_data(y_data, tracking_data_30, tracking_data_1, tracking_data_minus_1, player_play_data, player_data):
    """
    Generates y data for the ball carrier's location at 3 seconds (30 frames) after the snap.
    
    Args:
        tracking_data (pd.DataFrame): Player tracking data with frame-by-frame information.
    
    Returns:
        pd.DataFrame: A DataFrame with playId and the ball carrier's location (x, y) at 3 seconds after the snap.
    """
    
    tracking_data_30 = tracking_data_30.merge(player_play_data, on=["gameId", "playId", "nflId"], how="left")
    tracking_data_30 = tracking_data_30.merge(player_data, on=["nflId"], how='left')
    # y_data = []


    # Iterate through plays
    for game_id, game_df in tracking_data_30.groupby("gameId"):

        for play_id, play_df in game_df.groupby("playId"):
            # Extract the ball carrier's location
            ball_carrier = play_df[play_df["nflId"].isna()]  # True for the player with the ball
            pre_x = tracking_data_minus_1[(tracking_data_minus_1["gameId"] == game_id) & 
                    (tracking_data_minus_1["playId"] == play_id)].x.values
            pre_y = tracking_data_minus_1[(tracking_data_minus_1["gameId"] == game_id) & 
                    (tracking_data_minus_1["playId"] == play_id)].y.values
            x_1 = tracking_data_1[(tracking_data_1["gameId"] == game_id) & 
                    (tracking_data_1["playId"] == play_id)].x.values
            y_1 = tracking_data_1[(tracking_data_1["gameId"] == game_id) & 
                    (tracking_data_1["playId"] == play_id)].y.values
            if len(pre_x) == 0:
                continue
            if len (x_1) == 0:
                continue
            if not ball_carrier.empty:

                x = ball_carrier["x"].values[0]
                y = ball_carrier["y"].values[0]
                dx = x - pre_x[0]
                dy = y - pre_y[0]
                if x_1[0] > pre_x[0]:
                    dx *= -1
                    dy *= -1
                y_data.append({"game_id": game_id, "playId": play_id, "x": x, "y": y, "dx": dx, "dy": dy})

            
    # Convert to DataFrame
    print("Length: ", len(y_data))
    return y_data

In [ ]:
y_data = []
for i in range(9):
  week = i + 1
  print("week ", week)
  path = f"../data/tracking_week_{week}.csv"
  tracking_data = pd.read_csv(path)
  print(tracking_data.head())
  football_data = tracking_data[
          (tracking_data["club"] == "football") & 
          (tracking_data["nflId"].isna()) & 
          (tracking_data["jerseyNumber"].isna())
          
      ]
  tracking_data_30 = reduce_tracking_data_direct(football_data, 30)
  print(len(tracking_data_30))
  tracking_data_1 = reduce_tracking_data_direct(football_data, 1)
  print(len(tracking_data_1))

  tracking_data_minus_1 = reduce_tracking_data_direct(football_data, -1)
  print(len(tracking_data_minus_1))

  y_data = generate_y_data(y_data, tracking_data_30, tracking_data_1, tracking_data_minus_1, player_play_data, player_data)
